Predecir niveles de ruido cerca de aeropuerto.
Evaluar impacto acústico en zonas cercanas.
Datos de Dirección General de Aeronáutica Civil (DGAC)

Preguntas clave:
- ¿Qué factores influyen más en el ruido?
- ¿Hay diferencias según la aerolínea, hora o tipo de avión?
- ¿Por qué los sensores registran más ruido en las noches?


Diccionario de Variables
- NMT : Noise Monitoring Terminal: código del punto o estación de monitoreo de ruido.
- TLASmax : Timestamp de LASmax: fecha y hora de medición del nivel máximo de ruido.
- LASmax : Nivel máximo instantáneo de presión sonora ponderado A (dBA). Representa el mayor pico de ruido registrado durante el paso de una aeronave.
- SEL : Sound Exposure Level (dBA): Nivel de exposición sonora total de un evento de ruido, considerando duración y energía.
- Leq : Nivel equivalente (dBA): nivel continuo de presión sonora con la misma energía que el ruido fluctuante durante un período.
- LAZ : Nivel de ruido ponderado A con corrección de altitud o distancia (dBA). Es menos común, pero puede ser una medida ajustada.
- EPNL : Effective Perceived Noise Level (dB EPNL): Nivel de ruido percibido ajustado por la duración y características de la aeronave, usado en certificación aeronáutica.
- Temperature [°C] : Temperatura ambiental en grados Celsius. Puede influir en la propagación del sonido.
- A/D : Arrival / Departure: indica si el vuelo está aterrizando o despegando.
- Runway : Pista utilizada por la aeronave.
- Flight : Número de vuelo de la aeronave.
- Airline : Código IATA o ICAO de la aerolínea (ej: LAN, SKY).
- Airline (Name) : Nombre completo de la aerolínea.
- From/To : Ciudad o aeropuerto de origen o destino. Depende si es llegada o salida.
- Callsign : Identificador de radio usado por la aeronave.
- Aircraft Type : Tipo de aeronave.



EDA

Los sensores a demás de captar los eventos de ruido de los aviones, tambien capta ruido generado por otros agentes.
Es decir, los nulos no representan ningun evento otorgado por una aeronave.

In [4]:
#descargar datos de los sensores
import pandas as pd

df1 = pd.read_excel('../data/raw/NOISE TMR1.xlsx', skiprows=3)
df2 = pd.read_excel('../data/raw/NOISE TMR2.xlsx', skiprows=3)
df3 = pd.read_excel('../data/raw/NOISE TMR3.xlsx', skiprows=3)

In [5]:
#borrar primera fila vacia
df1 = df1.drop(index=0)
df2 = df2.drop(index=0)
df3 = df3.drop(index=0)

In [6]:
#Cuantos nulos hay en cada df
nulos1 = df1.isnull().sum().sum()
nulos2 = df2.isnull().sum().sum()
nulos3 = df3.isnull().sum().sum()
print(f'Hay {nulos1} nulos en el df1')
print(f'Hay {nulos2} nulos en el df2')
print(f'Hay {nulos3} nulos en el df3')

Hay 295228 nulos en el df1
Hay 200272 nulos en el df2
Hay 569901 nulos en el df3


In [7]:
#borrar nulos
df1 = df1.dropna()
df2 = df2.dropna()
df3 = df3.dropna()

In [8]:
#Eliminar columnas innecesarias
df1 = df1.drop(columns=['Flight', 'Callsign'])
df2 = df2.drop(columns=['Flight', 'Callsign'])
df3 = df3.drop(columns=['Flight', 'Callsign'])

In [9]:
#Transformar TLASmax en datetime
df1['TLASmax'] = pd.to_datetime(df1['TLASmax'], format='%d/%m/%Y %H:%M:%S')
df2['TLASmax'] = pd.to_datetime(df2['TLASmax'], format='%d/%m/%Y %H:%M:%S')
df3['TLASmax'] = pd.to_datetime(df3['TLASmax'], format='%d/%m/%Y %H:%M:%S')

In [10]:
#Agregar columna Hour y DayOfWeek y Date 
df1['Hour'] = df1['TLASmax'].dt.hour
df1['DayOfWeek'] = df1['TLASmax'].dt.dayofweek
df1['Date'] = df1['TLASmax'].dt.date
df2['Hour'] = df2['TLASmax'].dt.hour
df2['DayOfWeek'] = df2['TLASmax'].dt.dayofweek
df2['Date'] = df2['TLASmax'].dt.date
df3['Hour'] = df3['TLASmax'].dt.hour
df3['DayOfWeek'] = df3['TLASmax'].dt.dayofweek
df3['Date'] = df3['TLASmax'].dt.date

In [11]:
#Eliminar TLASmax
df1 = df1.drop(columns=['TLASmax'])
df2 = df2.drop(columns=['TLASmax'])
df3 = df3.drop(columns=['TLASmax'])

In [12]:
#Eliminar las otras columnas de ruido
df1 = df1.drop(columns=['SEL', 'Leq', 'LAZ','EPNL'])
df2 = df2.drop(columns=['SEL', 'Leq', 'LAZ','EPNL'])
df3 = df3.drop(columns=['SEL', 'Leq', 'LAZ','EPNL'])

In [13]:
#Cambiar nombre de la columna Temperature [°C] a Temp
df1 = df1.rename(columns={'Temperature [°C]': 'Temp'})
df2 = df2.rename(columns={'Temperature [°C]': 'Temp'})
df3 = df3.rename(columns={'Temperature [°C]': 'Temp'})

In [14]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 30894 entries, 1 to 75512
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   NMT             30894 non-null  object 
 1   LASmax          30894 non-null  float64
 2   Temp            30894 non-null  float64
 3   A/D             30894 non-null  object 
 4   Runway          30894 non-null  object 
 5   Airline         30894 non-null  object 
 6   Airline (Name)  30894 non-null  object 
 7   From/To         30894 non-null  object 
 8   Aircraft Type   30894 non-null  object 
 9   Hour            30894 non-null  int32  
 10  DayOfWeek       30894 non-null  int32  
 11  Date            30894 non-null  object 
dtypes: float64(2), int32(2), object(8)
memory usage: 2.8+ MB


In [15]:
#definir target y features
target = 'LASmax'
features = ['Hour', 'DayOfWeek','Hour','Temp','A/D','Runway','Airline','From/To','Aircraft Type']

In [16]:
from sklearn.model_selection import train_test_split
#Dividir en train y test
X1 = df1[features]
y1 = df1[target]
X2 = df2[features]
y2 = df2[target]
X3 = df3[features]
y3 = df3[target]
for col in ['A/D', 'Runway', 'Airline', 'From/To', 'Aircraft Type']:
    X1[col], _ = pd.factorize(X1[col])
    X2[col], _ = pd.factorize(X2[col])
    X3[col], _ = pd.factorize(X3[col])
X_train1, X_test1, y_train1, y_test1 = train_test_split(X1, y1, test_size=0.2, random_state=42)
X_train2, X_test2, y_train2, y_test2 = train_test_split(X2, y2, test_size=0.2, random_state=42)
X_train3, X_test3, y_train3, y_test3 = train_test_split(X3, y3, test_size=0.2, random_state=42)
#Guardar en csv
X_train1.to_csv('../data/processed/X_train1.csv', index=False)
X_test1.to_csv('../data/processed/X_test1.csv', index=False)
y_train1.to_csv('../data/processed/y_train1.csv', index=False)
y_test1.to_csv('../data/processed/y_test1.csv', index=False)
X_train2.to_csv('../data/processed/X_train2.csv', index=False)
X_test2.to_csv('../data/processed/X_test2.csv', index=False)
y_train2.to_csv('../data/processed/y_train2.csv', index=False)
y_test2.to_csv('../data/processed/y_test2.csv', index=False)
X_train3.to_csv('../data/processed/X_train3.csv', index=False)
X_test3.to_csv('../data/processed/X_test3.csv', index=False)
y_train3.to_csv('../data/processed/y_train3.csv', index=False)
y_test3.to_csv('../data/processed/y_test3.csv', index=False)

/tmp/ipykernel_720/1652166648.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X1[col], _ = pd.factorize(X1[col])
/tmp/ipykernel_720/1652166648.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X2[col], _ = pd.factorize(X2[col])
/tmp/ipykernel_720/1652166648.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_gu

In [17]:
#probar modelos random forest
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
#Entrenar modelo
rf1 = RandomForestRegressor(n_estimators=100, random_state=42)
rf1.fit(X_train1, y_train1)
rf2 = RandomForestRegressor(n_estimators=100, random_state=42)
rf2.fit(X_train2, y_train2)
rf3 = RandomForestRegressor(n_estimators=100, random_state=42)
rf3.fit(X_train3, y_train3)

RandomForestRegressor(random_state=42)

In [18]:
X_train3.head()

,Hour,DayOfWeek,Hour,Temp,A/D,Runway,Airline,From/To,Aircraft Type
48666,21,5,21,8.9,0,0,4,31,4
121507,11,3,11,17.2,0,1,0,24,6
88166,8,2,8,8.0,0,0,5,10,1
49994,5,3,5,7.8,0,0,21,3,14
79659,11,2,11,14.1,0,0,0,6,0


In [19]:
#predecir
y_pred1 = rf1.predict(X_test1)
y_pred2 = rf2.predict(X_test2)
y_pred3 = rf3.predict(X_test3)
#calcular error
mse1 = mean_squared_error(y_test1, y_pred1)
mse2 = mean_squared_error(y_test2, y_pred2)
mse3 = mean_squared_error(y_test3, y_pred3)
r2_1 = r2_score(y_test1, y_pred1)
r2_2 = r2_score(y_test2, y_pred2)
r2_3 = r2_score(y_test3, y_pred3)
print('MSE1:', mse1)
print('MSE2:', mse2)
print('MSE3:', mse3)
print('R2_1:', r2_1)
print('R2_2:', r2_2)
print('R2_3:', r2_3)

MSE1: 23.081033078761582
MSE2: 3.0200956210895633
MSE3: 7.308318638173924
R2_1: 0.14377187023376947
R2_2: 0.823236532781175
R2_3: 0.6063620621258735


In [21]:
from sklearn.model_selection import GridSearchCV
#Definir el modelo
#rf1
rf2
#rf3
#Definir los parametros
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}
#Definir el grid search
#grid_search1 = GridSearchCV(estimator=rf1, param_grid=param_grid, cv=3, n_jobs=-1, verbose=2)
grid_search2 = GridSearchCV(estimator=rf2, param_grid=param_grid, cv=3, n_jobs=-1, verbose=2)
#grid_search3 = GridSearchCV(estimator=rf3, param_grid=param_grid, cv=3, n_jobs=-1, verbose=2)
#Entrenar el grid search
#grid_search1.fit(X_train1, y_train1)
grid_search2.fit(X_train2, y_train2)
#grid_search3.fit(X_train3, y_train3)
#Imprimir los mejores parametros
#print('Mejores parametros 1:', grid_search1.best_params_)
print('Mejores parametros 2:', grid_search2.best_params_)
#print('Mejores parametros 3:', grid_search3.best_params_)

Fitting 3 folds for each of 24 candidates, totalling 72 fits
[CV] END max_depth=None, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=  10.7s
[CV] END max_depth=None, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=  10.8s
[CV] END max_depth=None, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=  11.3s
[CV] END max_depth=None, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=  22.2s
[CV] END max_depth=None, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=  23.6s
[CV] END max_depth=None, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=   9.5s
[CV] END max_depth=None, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=  23.1s
[CV] END max_depth=None, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=   9.7s
[CV] END max_depth=None, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=   9.6s
[CV] END max

/home/vscode/.local/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV] END max_depth=None, min_samples_leaf=2, min_samples_split=5, n_estimators=200; total time=  18.9s
[CV] END max_depth=10, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   5.4s
[CV] END max_depth=10, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=  10.1s
[CV] END max_depth=10, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=  10.3s
[CV] END max_depth=10, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=   4.7s
[CV] END max_depth=10, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=  10.7s
[CV] END max_depth=10, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=   5.2s
[CV] END max_depth=10, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=   5.4s
[CV] END max_depth=10, min_samples_leaf=1, min_samples_split=5, n_estimators=200; total time=  10.2s
[CV] END max_depth=10, min_samples_leaf=1, min_samples_split=5, n_estimators=200; total t

PicklingError: Could not pickle the task to send it to the workers.

Mejores parametros 1: {'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}